# Lab 18 — Ensemble Techniques, Part 1
## Problem Statement

A food-processing facility needs to sort rice grains into two varieties:
**Cammeo** and **Osmancik**. A camera captures each grain, and image-processing
software extracts measurements of its size and shape. Incorrect classifications
send grains to the wrong variety bin.

Build and compare classification models for this sorting task using **3,810 real
rice-grain measurements** from the UCI Rice dataset. Each record has seven
numeric features and a known variety label: **0 = Cammeo, 1 = Osmancik**.
The image measurements are already extracted and ready to use.

## Your Tasks

1. Prepare the features, labels, and training, validation, and test sets.
2. Fit individual models and combine their predictions using voting and stacking.
3. Build bagging, Random Forest, and Extra Trees classifiers.
4. Inspect bootstrap samples, reconstruct an out-of-bag prediction, and investigate model settings.
5. Compare validation results, evaluate the selected model on the test set, and recommend how it could be assessed for use in the sorter.

### Instructions

Open the notebook in Google Colab and use a CPU runtime. Run cells in order.
Complete all 12 TODOs by replacing the dots and removing each raise line.
Answer the questions, retain your outputs, and keep the final test set untouched
until Section 12.

## 1. Setup

The logistic-regression and nearest-neighbor pipelines scale the inputs.
The tree models use the original numeric features. Keeping scaling inside each
pipeline also keeps it within the appropriate training folds during stacking.

This notebook uses scikit-learn 1.4 or later. If a package is missing, run
the following in a new cell, then restart the runtime if Colab requests it:

    %pip install "scikit-learn>=1.4,<2" numpy pandas scipy matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone

print("scikit-learn version:", sklearn.__version__)

In [ ]:
from sklearn.ensemble import VotingClassifier, StackingClassifier, BaggingClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

plt.rcParams.update({"figure.figsize": (8, 4), "font.size": 11})
class_names = np.array(["Cammeo", "Osmancik"])

## 2. Load the dataset

The original [UCI Rice dataset](https://archive.ics.uci.edu/dataset/545/rice+cammeo+and+osmancik)
contains measurements from photographs of two rice varieties. No login or API
key is required. Run the cell below to download the ZIP archive and read the
ARFF table inside it.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile
from io import StringIO
from scipy.io import arff

url = "https://archive.ics.uci.edu/static/public/545/rice+cammeo+and+osmancik.zip"
archive = Path("rice.zip")
if not archive.exists():
    urlretrieve(url, archive)
with ZipFile(archive) as files:
    text = files.read("Rice_Cammeo_Osmancik.arff").decode("utf-8")
records, metadata = arff.loadarff(StringIO(text))
data = pd.DataFrame(records)

In [ ]:
data["Class"] = data["Class"].str.decode("utf-8")
print("Shape:", data.shape)
print("Missing values:", data.isna().sum().sum())
print("Varieties:", data["Class"].value_counts().to_dict())
display(data.head())

### Features

| Feature | Meaning |
|---|---|
| Area | Grain area in image pixels |
| Perimeter | Length around the grain boundary |
| Major_Axis_Length | Length of the fitted ellipse's long axis |
| Minor_Axis_Length | Length of its short axis |
| Eccentricity | How elongated the fitted ellipse is |
| Convex_Area | Area of the grain's convex hull |
| Extent | Grain area divided by bounding-box area |

These features are available from the image before classification. The variety
label is the answer and must stay out of the inputs. Several shape measurements
are related, which will matter when we interpret feature importance.

### TODO 1

Create X from the seven input columns and map Cammeo to 0 and Osmancik to 1. Reserve 20% for testing, then use 25% of the remaining development rows for validation. Preserve class proportions with stratify.

In [ ]:
X = ...
y = ...
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=..., stratify=y, random_state=42
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_dev, y_dev, test_size=..., stratify=y_dev, random_state=42
)
raise NotImplementedError("Complete TODO 1, then remove this line.")
print("Train, validation, test rows:", len(X_train), len(X_valid), len(X_test))
display(X_train.describe().round(2))

Use training data to fit models and validation data to compare them. The final
test set will be used once after selection. Stratification preserves the variety
proportions approximately; the split is 60% / 20% / 20%.

**Model selection rule:** choose the highest validation balanced accuracy.
Break a tie using validation accuracy, then model name. Balanced accuracy is the
average recall of the two varieties, so each variety counts equally.

The file has no production-batch or camera identifiers. A random grain holdout
checks performance within this source; testing new batches, lighting conditions,
and cameras would be needed before using a sorter in production.

In [ ]:
for label, color in [(0, "#bd7c35"), (1, "#287da8")]:
    rows = X_train.loc[y_train.eq(label)]
    plt.scatter(rows["Major_Axis_Length"], rows["Minor_Axis_Length"],
                s=10, alpha=0.30, color=color, label=class_names[label])
plt.xlabel("Major axis length")
plt.ylabel("Minor axis length")
plt.title("Actual training grains: two varieties overlap")
plt.legend()
plt.tight_layout()
plt.show()

**Question:** Why does the plot not establish how accurate a classifier will be?

**Answer:** _Write your answer here._

## 3. Establish individual models

Fit logistic regression, seven-neighbor classification, and a decision tree.
These models use different decision rules for the same task. Compare their
results with the ensembles to check whether combining models improves predictions.

The tree has min_samples_leaf=2, which retains flexibility while preventing
single-row leaves. No feature scaling is needed for the tree.

### TODO 2

Complete the fit and score calls for each model. Compare training and validation accuracy, then explain which model has the largest gap.

In [ ]:
base_models = [
    ("Logistic", make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))),
    ("KNN", make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))),
    ("Tree", DecisionTreeClassifier(min_samples_leaf=2, random_state=42))
]
models = {}
for name, model in base_models:
    ...
    train_accuracy = ...
    valid_accuracy = ...
    raise NotImplementedError("Complete TODO 2, then remove this line.")
    models[name] = model
    print(name, "train accuracy:", round(train_accuracy, 3),
          "validation accuracy:", round(valid_accuracy, 3))

**Question:** Which model has the largest training–validation gap? What does that suggest?

**Answer:** _Write your answer here._

## 4. Voting: choose a rule for combining predictions

**Hard voting** counts predicted labels. **Soft voting** averages the class
probabilities, then chooses the class with the larger average. Equal model weights
are used here. Soft voting can be affected by overconfident or poorly calibrated
members, so it is not automatically better.

VotingClassifier clones and fits the supplied estimators. Even though we fitted
the individual baselines, the voting objects fit their own members.

### TODO 3

Create and fit a hard voter and a soft voter using base_models.

In [ ]:
hard_vote = ...
soft_vote = ...
...
...
raise NotImplementedError("Complete TODO 3, then remove this line.")
models["Hard voting"], models["Soft voting"] = hard_vote, soft_vote

### Inspect one real grain

The code below finds the first validation grain where the two voting rules
disagree. It uses predictions, not the true label, to select the example.
If there is no disagreement in another software version, it uses the first grain.

In [ ]:
hard_predictions = hard_vote.predict(X_valid)
soft_predictions = soft_vote.predict(X_valid)
different = np.flatnonzero(hard_predictions != soft_predictions)
case_position = int(different[0]) if len(different) else 0
grain = X_valid.iloc[[case_position]]
probabilities = np.array([m.predict_proba(grain)[0] for m in soft_vote.estimators_])
labels = np.array([m.predict(grain)[0] for m in hard_vote.estimators_])
vote_table = pd.DataFrame({"Model": ["Logistic", "KNN", "Tree"],
                          "P(Osmancik)": probabilities[:, 1], "Class": labels})
print("Number of validation disagreements:", len(different))
display(vote_table.round(3))

### TODO 4

Calculate the hard answer from labels and the soft answer from probabilities. NumPy reminder: bincount counts labels, mean(axis=0) averages down rows, and argmax returns the position of the largest value.

In [ ]:
hard_answer = ...  # Class with the largest count.
mean_probabilities = ...  # Average across the three models.
soft_answer = ...  # Class with the largest average probability.
raise NotImplementedError("Complete TODO 4, then remove this line.")
print("Hard vote:", class_names[hard_answer])
print("Soft vote:", class_names[soft_answer])
print("Actual variety:", class_names[y_valid.iloc[case_position]])

In [ ]:
plt.bar(vote_table["Model"], vote_table["P(Osmancik)"], color="#6c9bb5")
plt.axhline(0.5, color="#555555", linestyle=":", label="Class boundary")
plt.axhline(mean_probabilities[1], color="#bd642e", label="Soft-vote mean")
plt.ylabel("Model probability for Osmancik")
plt.ylim(0, 1)
plt.title("One real grain: label counts and probability averages can disagree")
plt.legend()
plt.tight_layout()
plt.show()

**Question:** What information does hard voting discard?

**Answer:** _Write your answer here._

## 5. Stacking: learn the combining step

Voting uses a rule. Stacking uses a **meta-model**. Here, a logistic-regression
meta-model receives three probability columns, one from each base model.

Complete the stacking configuration with this training procedure in mind:

1. Split the training data into three folds.
2. Fit the base models without one fold and predict that held-out fold.
3. Repeat so every meta-training row uses base-model predictions made without that row.
4. Fit the meta-model on these out-of-fold predictions.
5. Refit the base models on the full training set for future predictions.

StackingClassifier handles this process. Validation and test records are not
used to train the meta-model. Scaling stays inside each base pipeline.

### TODO 5

Use the three-fold splitter defined below, a logistic-regression meta-model, and stack_method="predict_proba". Fit the stack using training rows only. Keep passthrough=False so the meta-model receives only the base predictions.

In [ ]:
folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
stacking = StackingClassifier(
    estimators=base_models,
    final_estimator=...,
    cv=..., stack_method=..., passthrough=False
)
...
raise NotImplementedError("Complete TODO 5, then remove this line.")
models["Stacking"] = stacking
display(pd.DataFrame(stacking.transform(X_valid.iloc[:5]),
                     columns=["Logistic P(1)", "KNN P(1)", "Tree P(1)"]))

The table shows the meta-model inputs for five **validation** grains at prediction
time. It is not the out-of-fold training table. For this binary task, scikit-learn
keeps one probability per base model because the two class probabilities sum to one.

**Question:** Why not fit the meta-model using base-model predictions on rows those base models already fitted?

**Answer:** _Write your answer here._

## 6. Bootstrap sampling: change the training rows

A bootstrap sample draws rows **with replacement**. A row can appear more than
once, and other rows may be absent. Sampling is not just shuffling.
Keep each selected row's features and correct label together.

### TODO 6

Use rng.choice to select len(X_train) positions with replace=True. Use the same positions with .iloc to select matching X and y rows.

In [ ]:
rng = np.random.default_rng(42)
sample_positions = ...
X_boot = ...
y_boot = ...
raise NotImplementedError("Complete TODO 6, then remove this line.")
unique_positions = np.unique(sample_positions)
omitted_positions = np.setdiff1d(np.arange(len(X_train)), unique_positions)
print("Selections:", len(sample_positions), "Unique rows:", len(unique_positions))
print("Repeated selections:", len(sample_positions) - len(unique_positions))
print("Rows left out:", len(omitted_positions))

In [ ]:
selection_counts = np.bincount(sample_positions, minlength=len(X_train))
display(pd.DataFrame({
    "Training position": np.arange(12),
    "Times selected": selection_counts[:12],
    "Actual variety": class_names[y_train.iloc[:12].to_numpy()]
}))
bootstrap_tree = DecisionTreeClassifier(min_samples_leaf=2, random_state=42)
bootstrap_tree.fit(X_boot, y_boot)
print("One bootstrap tree's validation accuracy:", round(bootstrap_tree.score(X_valid, y_valid), 3))

**Question:** Did selecting a grain twice create a new grain? Can a left-out row be used to check this tree?

**Answer:** _Write your answer here._

## 7. Bagging: fit separate trees and combine predictions

Train 120 flexible trees on different bootstrap samples. Each sample makes 80%
as many row selections as there are training rows; repeats still count as selections.
Every tree can use all seven input columns.

**Prediction rule:** with decision trees, BaggingClassifier averages **class probabilities**,
then selects the largest average. RandomForestClassifier and ExtraTreesClassifier
also average class probabilities. We do not average their branches.

### TODO 7

Fit the bagger with 120 trees, bootstrap=True, max_samples=0.8, and all feature columns.

In [ ]:
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(min_samples_leaf=2),
    n_estimators=..., max_samples=..., max_features=1.0,
    bootstrap=..., bootstrap_features=False, random_state=42, n_jobs=2
)
...
raise NotImplementedError("Complete TODO 7, then remove this line.")
models["Bagging"] = bagging

In [ ]:
first_sample = bagging.estimators_samples_[0]
print("Selections in the first bag:", len(first_sample))
print("Distinct training rows:", len(np.unique(first_sample)))
sample_counts = np.array([
    np.bincount(rows, minlength=len(X_train))[:20]
    for rows in bagging.estimators_samples_[:3]
])
plt.imshow(sample_counts, cmap="Blues", aspect="auto", vmin=0)
plt.yticks([0, 1, 2], ["Tree 1", "Tree 2", "Tree 3"])
plt.xticks(range(20), range(20))
plt.xlabel("Training-row position (first 20 shown)")
plt.colorbar(ticks=range(sample_counts.max() + 1), label="Times selected for this tree")
plt.title("Real bootstrap samples: zero means out of bag")
plt.tight_layout()
plt.show()

**Question:** Why do these samples have fewer selections than the earlier manual bootstrap?

**Answer:** _Write your answer here._

## 8. Random Forest: randomize feature choices at each node

Keep row bootstrapping, but offer each split a random subset of features. With
seven inputs and max_features=2, a node normally considers two feature choices.
A later node receives a fresh subset. This does not remove the other columns
from the whole tree.

| Setting used here | Meaning |
|---|---|
| n_estimators=120 | Fit 120 trees |
| max_features=2 | Consider two feature choices at each split |
| bootstrap=True | Sample training rows with replacement |
| max_samples=0.8 | Make about 80% of the training-row count in selections per tree |
| min_samples_leaf=2 | Prevent one-row leaves in each tree |
| n_jobs=2 | Request two parallel workers; this changes scheduling, not the split criterion |
| oob_score=True | Also estimate accuracy from tree-specific left-out rows |

BaggingClassifier's max_features selects columns **for each whole estimator**.
RandomForestClassifier's max_features limits choices **at each split**.

### TODO 8

Complete the forest settings shown in the table, fit it, and add it to models.

In [ ]:
forest = RandomForestClassifier(
    n_estimators=..., max_features=..., bootstrap=..., max_samples=...,
    min_samples_leaf=2, oob_score=True, random_state=42, n_jobs=2
)
...
raise NotImplementedError("Complete TODO 8, then remove this line.")
models["Random Forest"] = forest

### Do the trees keep starting with the same feature?

Count the actual winning root feature in each fitted tree. This shows the
chosen features, not the complete candidate sets the algorithm considered.
The bagger's roots are mapped back through its stored feature-column indices.

In [ ]:
bag_roots = [X_train.columns[cols[t.tree_.feature[0]]]
             for t, cols in zip(bagging.estimators_, bagging.estimators_features_)]
forest_roots = [X_train.columns[t.tree_.feature[0]] for t in forest.estimators_]
root_counts = pd.DataFrame({
    "Bagging": pd.Series(bag_roots).value_counts(),
    "Random Forest": pd.Series(forest_roots).value_counts()
}).fillna(0).astype(int)
root_counts.plot.barh(color=["#b98343", "#427e9c"])
plt.xlabel("Number of trees choosing this root feature")
plt.title("Random feature choices encourage different first questions")
plt.tight_layout()
plt.show()

### Useful members need different mistakes

Compare the first 20 members from each ensemble on the
same validation grains. For each member, an error is 1 when it is wrong and 0
when it is correct. Correlation near 1 means two members tend to fail together.
We average the pairwise error correlations and also report average member accuracy.

This is a small empirical diagnostic. Lower correlation alone does not make
an ensemble better: the members must also be useful.

In [ ]:
bag_member_predictions = np.column_stack([
    t.predict(X_valid.to_numpy()[:, cols])
    for t, cols in zip(bagging.estimators_[:20], bagging.estimators_features_[:20])
])
forest_member_predictions = np.column_stack([
    t.predict(X_valid.to_numpy()) for t in forest.estimators_[:20]
])
diversity_rows = []
for name, predictions in [("Bagging", bag_member_predictions), ("Random Forest", forest_member_predictions)]:
    errors = predictions != y_valid.to_numpy()[:, None]
    correlations = pd.DataFrame(errors.astype(int)).corr().to_numpy()
    average_correlation = np.nanmean(correlations[np.triu_indices(20, k=1)])
    diversity_rows.append([name, 1 - errors.mean(), average_correlation])
display(pd.DataFrame(diversity_rows, columns=["Model", "Mean member accuracy", "Mean error correlation"]).round(3))

**Question:** Would 120 identical copies of the same tree give the same benefit?

**Answer:** _Write your answer here._

### Familiar controls still apply

Each tree still supports max_depth, min_samples_split, min_samples_leaf,
max_leaf_nodes, min_impurity_decrease, criterion, and ccp_alpha. Class weights
change class influence and can help with imbalance, but do not guarantee good
minority-class recall. These controls act on each member.

For example, a node has six rows; min_samples_split=4 and min_samples_leaf=2.
A proposed split creates children of sizes one and five. The parent can be
considered for splitting, but that proposed split is invalid.

More trees require more computation. A smaller max_features can increase variety
but can also hide useful inputs too often. With bootstrap=False, use max_samples=None;
the trees use all rows and ordinary OOB scoring is unavailable.

## 9. OOB: which trees are allowed to predict one training grain?

A grain is out of bag for a tree only when that tree's bootstrap sample omitted
it. For that grain's OOB estimate, use **only those trees**.

The loop scaffold inspects the actual stored samples. Use training position 0
and collect class probabilities only from eligible trees. Individual forest members
were fitted internally on arrays, so we pass an array when inspecting them.

### TODO 9

Complete the eligibility condition so a tree contributes only when it did not train on the selected row. Average those probabilities, choose the class, and check your answer against sklearn.

In [ ]:
row_position = 0
training_grain = X_train.iloc[[row_position]].to_numpy()
oob_probabilities = []
eligible_trees = []
for number, (tree, rows) in enumerate(zip(forest.estimators_, forest.estimators_samples_)):
    if ...:
        eligible_trees.append(number)
        oob_probabilities.append(tree.predict_proba(training_grain)[0])
oob_mean = ...
oob_answer = ...
raise NotImplementedError("Complete TODO 9, then remove this line.")
print("Eligible trees:", len(eligible_trees), "of", len(forest.estimators_))
print("Manual OOB probabilities:", oob_mean.round(3))
print("OOB answer:", class_names[oob_answer], "Actual:", class_names[y_train.iloc[row_position]])
print("Matches sklearn:", np.allclose(oob_mean, forest.oob_decision_function_[row_position]))

Now score all original training rows **once each**, after combining that row's
eligible predictions. A small forest can leave some rows without any OOB prediction;
check coverage rather than treating missing predictions as a class.

In [ ]:
oob_probabilities_all = forest.oob_decision_function_
covered = np.isfinite(oob_probabilities_all).all(axis=1) & (oob_probabilities_all.sum(axis=1) > 0)
oob_predictions = forest.classes_[oob_probabilities_all[covered].argmax(axis=1)]
print("OOB coverage:", covered.sum(), "/", len(y_train))
print("OOB accuracy:", round(accuracy_score(y_train.iloc[np.flatnonzero(covered)], oob_predictions), 3))
print("forest.oob_score_:", round(forest.oob_score_, 3))
print("Validation accuracy:", round(forest.score(X_valid, y_valid), 3))

**Question:** Can a tree vote on a grain it trained on to improve that grain’s OOB answer?

**Answer:** _Write your answer here._

## 10. Extra Trees: randomize the proposed thresholds

Random Forest searches for a good threshold among the selected features.
Extra Trees proposes random thresholds for the selected features and keeps the
best valid proposal. It does **not** accept every random proposal blindly.

For a clear comparison, keep the forest's 120 trees, feature count, bootstrap
sample size, and leaf-size control the same. The main change is the split strategy.
ExtraTreesClassifier normally defaults to bootstrap=False; we explicitly use
bootstrap=True here to match the Random Forest experiment.

### TODO 10

Fit ExtraTreesClassifier with the same settings shown for the forest.

In [ ]:
extra_trees = ExtraTreesClassifier(
    n_estimators=..., max_features=..., bootstrap=True, max_samples=0.8,
    min_samples_leaf=2, random_state=42, n_jobs=2
)
...
raise NotImplementedError("Complete TODO 10, then remove this line.")
models["Extra Trees"] = extra_trees

**Question:** Why did we override the Extra Trees bootstrap default?

**Answer:** _Write your answer here._

## 11. Investigate settings, then compare models

Run a controlled parameter experiment before inspecting the final test. Change
the number of trees while keeping two features per split; then change the number
of feature choices while keeping 120 trees. The other settings stay fixed.

This diagnostic helps explain parameter effects. The model benchmark that follows
compares the original preset models; this exercise is not an exhaustive search.

### TODO 11

Complete set_params, fit, and validation scoring for the five settings. Use the resulting table to write one observation about tree count and one about feature choices. Do not use test data.

In [ ]:
setting_results = []
for number_of_trees, feature_count in [(20, 2), (60, 2), (120, 2), (120, 1), (120, 7)]:
    experiment = clone(forest).set_params(
        n_estimators=..., max_features=..., oob_score=False
    )
    ...
    score = ...
    raise NotImplementedError("Complete TODO 11, then remove this line.")
    setting_results.append([number_of_trees, feature_count, score])
setting_table = pd.DataFrame(setting_results, columns=[
    "Trees", "Features per split", "Validation balanced accuracy"
])
display(setting_table.round(4))

**Question:** Did more trees always improve the score? What happened when all seven features were available?

**Answer:** _Write your answer here._

### Compare the preset models

Compare all models using the same training and validation split. Include the
individual models: a simpler model may perform as well as or better than an ensemble.

Read both training and validation results. A large gap suggests training-specific
fit, while a tiny validation difference may just reflect a few grains.

In [ ]:
results = []
for name, model in models.items():
    training_predictions = model.predict(X_train)
    predictions = model.predict(X_valid)
    results.append([name, accuracy_score(y_train, training_predictions),
                    balanced_accuracy_score(y_train, training_predictions),
                    accuracy_score(y_valid, predictions), balanced_accuracy_score(y_valid, predictions)])
comparison = pd.DataFrame(results, columns=[
    "Model", "Training accuracy", "Training balanced accuracy",
    "Validation accuracy", "Validation balanced accuracy"
])
ranking = comparison.sort_values(
    ["Validation balanced accuracy", "Validation accuracy", "Model"],
    ascending=[False, False, True]
)
display(ranking.round(4))

In [ ]:
ranking.set_index("Model")[["Training balanced accuracy", "Validation balanced accuracy"]].plot.barh(
    figsize=(9, 5), color=["#aebfca", "#447f9e"], xlim=(0, 1)
)
plt.xlabel("Balanced accuracy")
plt.gca().invert_yaxis()
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2, frameon=False, fontsize=9)
plt.title("Compare individual models and ensembles on the same split")
plt.tight_layout()
plt.show()

**Question:** Did every ensemble improve on every individual model? What would you recommend?

**Answer:** _Write your answer here._

### Which measurements helped the forest?

Measure the drop in validation balanced accuracy when
one feature is shuffled. It describes the fitted Random Forest, even if another
model wins the comparison. It does not use the test set.

Correlated features can substitute for one another, reducing an individual
feature's measured importance. Importance is model-dependent and is not causation.

In [ ]:
importance = permutation_importance(
    forest, X_valid, y_valid, scoring="balanced_accuracy",
    n_repeats=5, random_state=42, n_jobs=2
)
order = importance.importances_mean.argsort()
plt.barh(X_train.columns[order], importance.importances_mean[order],
         xerr=importance.importances_std[order], color="#43809d")
plt.axvline(0, color="#777777", linewidth=1)
plt.xlabel("Drop in validation balanced accuracy after shuffling")
plt.title("Random Forest permutation importance (± one repeat standard deviation)")
plt.tight_layout()
plt.show()

An importance below zero means that shuffling improved the score on average in
these repeats. It is not enough evidence to remove that feature automatically.
The error bars show variation across shuffles, not confidence intervals for
future production performance.

## 12. Lock the choice, then evaluate once on the final test

Use the validation rule stated at the start. Clone that model's settings and
refit on all development rows (training plus validation). Only now predict the
test labels. Do not use the test result to switch models or tune settings.

In [ ]:
best_name = ranking.iloc[0]["Model"]
print("Locked selection from validation:", best_name)
final_model = clone(models[best_name])

### TODO 12

Fit final_model on X_dev and y_dev, predict X_test, and report final classification scores. Use the confusion matrix below to write your final three-sentence recommendation and submit the notebook.

In [ ]:
...
test_predictions = ...
raise NotImplementedError("Complete TODO 12, then remove this line.")
print("Test accuracy:", round(accuracy_score(y_test, test_predictions), 3))
print("Test balanced accuracy:", round(balanced_accuracy_score(y_test, test_predictions), 3))
print("Misclassified grains:", np.sum(test_predictions != y_test.to_numpy()), "of", len(y_test))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions, display_labels=class_names, cmap="Blues", colorbar=False
)
plt.title("Final test: " + best_name)
plt.tight_layout()
plt.show()

The off-diagonal cells count grains routed to the wrong variety. These are errors
on this held-out dataset, not a guaranteed contamination rate for a real factory.
An operational evaluation would also check new production batches, camera consistency,
throughput, memory use, and how uncertain cases are sent for review.

## 13. Final Recommendation and Submission

Submit the executed notebook with all 12 TODOs, bootstrap counts, one manually
reconstructed OOB prediction, the parameter-experiment table, the validation
comparison, and the final confusion matrix.

**Final recommendation:** write three sentences
naming the selected model and its test result, explaining the wrong-variety
counts, and identifying one check needed before factory use.

Answer these questions using your results:

1. What changes between voting and stacking?
2. What are the two random choices in a Random Forest?
3. Can an in-bag tree contribute to a row's OOB answer?
4. What extra randomness does Extra Trees introduce?
5. Did the ensemble guarantee a better model in this experiment?
6. Why should we test new batches before using the sorter?

**Question:** Your final recommendation (three sentences)

**Answer:** _Write your answer here._

More trees often stabilize an average, but this single run does not measure prediction
variance or establish that scores must rise monotonically. Repeated samples
would be needed to study stability directly.

Random Forest offers flexible rules, little scaling preparation, and parallel
training. Costs include more computation, less direct interpretability than one
small tree, and possible difficulties with very wide sparse inputs or rare classes.

## Data credit and software references

- I. Cinar and M. Koklu (2019), *Classification of Rice Varieties Using Artificial
  Intelligence Methods*, International Journal of Intelligent Systems and
  Applications in Engineering, 7(3), 188–194.
  [Paper DOI](https://doi.org/10.18201/ijisae.2019355381).
  **Rice (Cammeo and Osmancik)**, UCI Machine Learning Repository:
  [dataset](https://archive.ics.uci.edu/dataset/545/rice+cammeo+and+osmancik) ·
  [dataset DOI: 10.24432/C5MW4Z](https://doi.org/10.24432/C5MW4Z) ·
  [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
  Lab transformations: decode the variety name, map it to 0/1, and split rows.
  No synthetic observations are generated.
- scikit-learn:
  [VotingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html),
  [StackingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html),
  [BaggingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html),
  [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html),
  [ExtraTreesClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.ExtraTreesClassifier.html),
  [permutation importance](https://scikit-learn.org/stable/modules/permutation_importance.html).